# OpenUBA SDK: Visualizations

This notebook demonstrates creating visualizations with real rendered output using multiple backends.

**SDK functions covered:** `create_visualization` (with `figure=`), `render`, `publish_visualization`, `list_visualizations`

In [ ]:
import openuba
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print(f'OpenUBA SDK v{openuba.__version__}')

## 1. Matplotlib: Risk Score Distribution

Pass a matplotlib figure directly — the SDK renders it to SVG and uploads automatically.

In [ ]:
viz_ids = []

# matplotlib: risk score distribution
fig, ax = plt.subplots(figsize=(10, 5))
np.random.seed(42)
risk_scores = np.random.beta(2, 5, 1000)
ax.hist(risk_scores, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.set_title('Entity Risk Score Distribution')
ax.set_xlabel('Risk Score')
ax.set_ylabel('Count')
ax.axvline(x=0.7, color='red', linestyle='--', label='High Risk Threshold')
ax.legend()

viz = openuba.create_visualization(
    name='nb-risk-score-distribution',
    backend='matplotlib',
    description='Risk Score Distribution using matplotlib',
    figure=fig,
)
viz_ids.append(viz['id'])
print(f"Created: {viz['name']} (has rendered output: {viz.get('rendered_output') is not None})")
plt.close(fig)

## 2. Seaborn: Login Activity Heatmap

In [ ]:
import seaborn as sns

df = pd.DataFrame({
    'hour': list(range(24)) * 7,
    'day': [d for d in ['Mon','Tue','Wed','Thu','Fri','Sat','Sun'] for _ in range(24)],
    'logins': np.random.poisson(10, 168),
})
pivot = df.pivot_table(values='logins', index='day', columns='hour', aggfunc='mean')

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(pivot, cmap='YlOrRd', ax=ax)
ax.set_title('Login Activity Heatmap')

viz = openuba.create_visualization(
    name='nb-login-heatmap',
    backend='seaborn',
    description='Login Activity Heatmap using seaborn',
    figure=fig,
)
viz_ids.append(viz['id'])
print(f"Created: {viz['name']} (has rendered output: {viz.get('rendered_output') is not None})")
plt.close(fig)

## 3. Plotly: Interactive Risk Timeline

In [ ]:
import plotly.graph_objects as go

fig_plotly = go.Figure()
fig_plotly.add_trace(go.Scatter(
    x=list(range(100)),
    y=np.cumsum(np.random.randn(100)),
    mode='lines',
    name='Risk Score Trend',
    line=dict(color='#00C49F'),
))
fig_plotly.update_layout(
    title='Entity Risk Over Time',
    xaxis_title='Time (hours)',
    yaxis_title='Cumulative Risk',
    template='plotly_dark',
)

viz = openuba.create_visualization(
    name='nb-risk-timeline',
    backend='plotly',
    description='Interactive Risk Timeline using plotly',
    figure=fig_plotly,
)
viz_ids.append(viz['id'])
print(f"Created: {viz['name']} (has rendered output: {viz.get('rendered_output') is not None})")

## 4. NetworkX: Lateral Movement Graph

In [ ]:
import networkx as nx

G = nx.DiGraph()
G.add_edges_from([
    ('workstation-1', 'dc-01'),
    ('workstation-1', 'fileserver'),
    ('dc-01', 'dc-02'),
    ('dc-02', 'db-server'),
    ('fileserver', 'backup-server'),
    ('workstation-2', 'dc-01'),
    ('workstation-3', 'fileserver'),
    ('dc-01', 'exchange'),
])

viz = openuba.create_visualization(
    name='nb-lateral-movement',
    backend='networkx',
    description='Lateral Movement Graph using networkx',
    figure=G,
)
viz_ids.append(viz['id'])
print(f"Created: {viz['name']} (has rendered output: {viz.get('rendered_output') is not None})")
plt.close('all')

## 5. Matplotlib: Anomaly Scatter Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
n = 200
x = np.random.randn(n) * 2 + 5
y = np.random.randn(n) * 2 + 5
anomaly_mask = np.random.random(n) > 0.9
ax.scatter(x[~anomaly_mask], y[~anomaly_mask], c='#82ca9d', alpha=0.6, label='Normal', s=30)
ax.scatter(x[anomaly_mask], y[anomaly_mask], c='#ff7300', alpha=0.9, label='Anomaly', s=80, marker='x')
ax.set_title('Login Behavior: Normal vs Anomalous')
ax.set_xlabel('Login Frequency (per day)')
ax.set_ylabel('Session Duration (hours)')
ax.legend()

viz = openuba.create_visualization(
    name='nb-anomaly-scatter',
    backend='matplotlib',
    description='Anomaly Scatter Plot using matplotlib',
    figure=fig,
)
viz_ids.append(viz['id'])
print(f"Created: {viz['name']} (has rendered output: {viz.get('rendered_output') is not None})")
plt.close(fig)

## 6. List & Publish Visualizations

In [ ]:
all_viz = openuba.list_visualizations()
print(f'Total visualizations: {len(all_viz)}')
for v in all_viz:
    has_output = 'rendered' if v.get('rendered_output') else 'empty'
    print(f'  {v["name"]:40s} backend={v["backend"]:12s} [{has_output}]')

# publish the first one
if viz_ids:
    try:
        pub = openuba.publish_visualization(viz_ids[0])
        print(f'\nPublished: {pub.get("name")} -> published={pub.get("published")}'	)
    except Exception as e:
        print(f'publish: {e}')

## 7. Using openuba.render() with auto-push

The `render()` function can also push output to an existing visualization record.

In [ ]:
# create an empty viz, then render+push separately
viz_empty = openuba.create_visualization(
    name='nb-render-push-demo',
    backend='matplotlib',
    description='Demo of render() with auto-push',
)

fig, ax = plt.subplots(figsize=(8, 4))
hours = range(24)
normal = [10, 5, 3, 2, 2, 3, 8, 25, 45, 40, 35, 30, 28, 32, 35, 38, 40, 35, 25, 20, 18, 15, 12, 11]
anomalous = [12, 8, 15, 22, 18, 5, 10, 28, 42, 38, 33, 31, 29, 30, 36, 39, 41, 33, 22, 19, 16, 14, 13, 10]
ax.plot(hours, normal, 'o-', color='#82ca9d', label='Normal Day')
ax.plot(hours, anomalous, 's--', color='#ff7300', label='Anomalous Day')
ax.fill_between(hours, normal, anomalous, alpha=0.1, color='red')
ax.set_title('Login Volume: Normal vs Anomalous Day')
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Login Count')
ax.legend()

# render() with viz_id auto-pushes to the platform
result = openuba.render(fig, viz_id=viz_empty['id'])
print(f"Rendered: type={result['type']}, content length={len(result['content'])}")
viz_ids.append(viz_empty['id'])
plt.close(fig)

## Summary

In [ ]:
print(f'=== VISUALIZATIONS COMPLETE ===')
print(f'Visualizations created: {len(viz_ids)}')
print(f'Backends used: matplotlib, seaborn, plotly, networkx')
print(f'')
print(f'Key patterns:')
print(f'  openuba.create_visualization(name, backend, figure=fig)  # create + render in one call')
print(f'  openuba.render(fig, viz_id=id)                           # render + push to existing viz')
print(f'  openuba.update_visualization(id, rendered_output=svg)    # manual update')